# Setup

In [1]:
%run notebook_setup.py

import pandas as pd
import numpy as np
from scipy.stats import trim_mean
import polars as pl
import pandas as pd
import logging
import gc
import random
import datetime as dt

import matplotlib.pyplot as plt
import seaborn as sns

from src.config.dir_config import OUTPUT_PATH_DEMAND_SUMMARY
from src.config.bigquery_config import CREDENTIALS_GBQ, PROJECT_ID_GBQ
from src.utils.read_data import read_data
from src.utils.setup_logging import setup_logging
from src.utils.create_week_date import add_week_start_date
from src.utils.export_as_excel import export_dataframes_as_tables


import logging

setup_logging()

Now you can import modules from the project root: /bi/workspace/Projects/Forecast/forecast


# Functions

In [2]:
def add_real_sales_sv(df):
    def var_forward_sum(g):
        g = g.sort_values("week_number")
        sales = g["weekly_sales"].fillna(0).to_numpy(int)
        weeks = g["semana_vta"].fillna(1).astype(int).to_numpy()
        n = len(sales)

        s = np.cumsum(sales)
        i = np.arange(n)
        right = np.minimum(i + weeks, n)
        s_pad = np.concatenate(([0.0], s))
        
        return pd.Series(s_pad[right] - s_pad[i], index=g.index)

    df["real_sales"] = (
    df
    .groupby(["cod_sucursal", "cod_producto", "cod_talla"], group_keys=False)[["week_number", "weekly_sales", "semana_vta"]]
    .apply(var_forward_sum)
    .astype("int16")
    )

    return df

# Data import

In [3]:
data_genex = pd.read_parquet('../data/processed/genex_data_processed.parquet')
demand_summary = pd.read_parquet('../data/processed/demand_summary.parquet')
data_sales = pd.read_parquet('../data/processed/weekly_sales_by_season/Invierno/weekly_sales_Invierno_2025_processed.parquet')
classifications = pd.read_excel('../data/external/Consolidado clasificaciones modelo BI.xlsx')
transfers = pd.read_parquet('../data/processed/trf_data_processed.parquet')

# Data processing

In [4]:
demand_summary = demand_summary[demand_summary['cod_sucursal'] != 767]
demand_summary = demand_summary[demand_summary['nombre_depto'] != 'Bolsas y bolsos']
demand_summary = demand_summary[demand_summary['nombre_depto'] != 'Miscelaneos']
demand_summary = demand_summary[demand_summary['nombre_temporada'] == 'Invierno']
demand_summary = demand_summary[demand_summary['ano_temporada'] == '2025']

In [5]:
demand_info = demand_summary[['cod_producto', 'cod_talla', 'cod_sucursal',
                              'nombre_sucursal',
                              'nombre_temporada','ano_temporada','nombre_depto','nombre_linea','nom_talla',
                              'demand_type']].copy()

In [6]:
transfers_pivot = transfers.pivot_table(
    index=['cod_sucursal','cod_producto','cod_talla','cod_ano_comercial','cod_semana'],
    observed=True,
    columns="nombre_razon_group",
    values= 'cantidad_des',
    aggfunc="sum",
    fill_value=0
).reset_index()

transfers_pivot.columns.name = None

In [7]:
transfers_pivot_summary = transfers.pivot_table(
    index=['cod_sucursal','cod_producto','cod_talla'],
    observed=True,
    columns="nombre_razon_group",
    values= 'cantidad_des',
    aggfunc="sum",
    fill_value=0
).reset_index()

transfers_pivot_summary.columns.name = None

In [8]:
demand_summary = demand_summary.merge(
    transfers_pivot_summary,
    on = ['cod_sucursal','cod_producto','cod_talla'],
    how = 'left'
)

demand_summary[["PREDISTRIBUIDA",
        "REPOSICION AUTOMATIC",
        "CARGA MANUAL",
        "OTRAS"]] = demand_summary[["PREDISTRIBUIDA",
                                    "REPOSICION AUTOMATIC",
                                    "CARGA MANUAL",
                                    "OTRAS"]].fillna(0)

## *A) Proccessing data_all*

In [9]:
data_all = data_sales.merge(data_genex.drop(columns=['cod_talla']),
                            on=['cod_producto','cod_sku', 'cod_sucursal', 'cod_ano_comercial','cod_semana'],
                                how='left')

data_all = data_all.merge(demand_info,
                            on=['cod_producto','cod_talla', 'cod_sucursal'],
                            how='left')

data_all = data_all.merge(classifications,
                            on=['nombre_temporada','cod_sucursal','nombre_depto', 'nombre_linea'],
                            how='left')

data_all = data_all.merge(transfers_pivot,
                            on=['cod_producto','cod_talla', 'cod_sucursal', 'cod_ano_comercial','cod_semana'],
                                how='left')

del demand_info, classifications, data_sales
gc.collect()

117

In [10]:
data_all = add_week_start_date(data_all,
                               year_col='cod_ano_comercial',
                               week_col='cod_semana')

In [11]:
data_all['clasificacion'] = pd.Categorical(data_all['clasificacion'],
                                           categories=['AA','A','B','C'],
                                           ordered=True)

In [12]:
data_all[["PREDISTRIBUIDA",
        "REPOSICION AUTOMATIC",
        "CARGA MANUAL",
        "OTRAS"]] = data_all[["PREDISTRIBUIDA",
                                    "REPOSICION AUTOMATIC",
                                    "CARGA MANUAL",
                                    "OTRAS"]].fillna(0)

In [13]:
data_all = data_all[data_all['cod_sucursal'] != 767]
data_all = data_all[data_all['nombre_depto'] != 'Bolsas y bolsos']
data_all = data_all[data_all['nombre_depto'] != 'Miscelaneos']
data_all = data_all[data_all['nombre_temporada'] == 'Invierno']
data_all = data_all[data_all['ano_temporada'] == '2025']

In [ ]:
dict_factor_l_dias = {
    2: 28/7,
    3: 28/14,
    4: 28/21,
}

# Condiciones
mask_fecha = data_all['week_start_date'].dt.date.isin([
    dt.date(2025, 5, 5),
    dt.date(2025, 4, 21)
])

mask_factores = data_all['week_number'].isin(dict_factor_l_dias.keys())

# Calculamos el nuevo vta_promedio en dos pasos
data_all['vta_promedio'] = np.where(
    mask_fecha & mask_factores,
    data_all['mean_sales_past_4_weeks'] * data_all['week_number'].map(dict_factor_l_dias),
    np.where(
        mask_fecha & ~mask_factores,
        data_all['mean_sales_past_4_weeks'],
        data_all['vta_promedio']
    )
)

# Redondeo al final (una sola vez)
data_all['vta_promedio'] = data_all['vta_promedio'].round(3)

In [ ]:
data_all['factor_l_dias'] = np.where(
    data_all['mean_sales_past_4_weeks'] == 0,
    np.nan,
    (data_all['vta_promedio'] / data_all['mean_sales_past_4_weeks']).round(3)
)

In [17]:
data_all = add_real_sales_sv(data_all)

# Export

In [18]:
data_all.to_parquet('../sandbox/data_genex_venta_transfer.parquet')

In [19]:
demand_summary.to_parquet('../sandbox/demand_summary.parquet')

---

# EDA

In [20]:
data_all = pd.read_parquet('../sandbox/data_genex_venta_transfer.parquet')

## Sample product

In [21]:
cod_sku = 641769518
cod_sucursal = 17

# Filtrar los datos base
data_sample = data_all[data_all['cod_sku'] == cod_sku].copy()
data_sample = data_sample[data_sample['cod_sucursal'] == cod_sucursal].reset_index(drop=True)


# Ver columnas de interés
data_sample[[
    'cod_sucursal','cod_producto','cod_sku','week_number','week_start_date',
    'can_final','repo_x_dda','weekly_sales', 'stock_start_week',
    'mean_sales_past_4_weeks','vta_promedio','factor_l_dias','factor','semana_vta',
]]

,cod_sucursal,cod_producto,cod_sku,week_number,week_start_date,can_final,repo_x_dda,weekly_sales,stock_start_week,mean_sales_past_4_weeks,vta_promedio,factor_l_dias,factor,semana_vta
0,17,641769,641769518,1,2025-04-14,NaN,NaN,1,12,0.00,NaN,NaN,NaN,NaN
1,17,641769,641769518,2,2025-04-21,2.0,0.0,4,17,0.25,1.000,4.000,2.5,8.0
2,17,641769,641769518,3,2025-04-28,0.0,0.0,21,19,1.25,2.500,2.000,1.0,4.0
3,17,641769,641769518,4,2025-05-05,47.0,78.0,20,12,6.50,8.667,1.333,1.3,8.0
4,17,641769,641769518,5,2025-05-12,0.0,18.0,0,11,11.50,11.500,1.000,0.7,6.0
5,17,641769,641769518,6,2025-05-19,0.0,0.0,1,11,11.25,11.250,1.000,0.7,6.0
6,17,641769,641769518,7,2025-05-26,0.0,68.0,6,46,10.50,10.500,1.000,1.8,6.0
7,17,641769,641769518,8,2025-06-02,0.0,0.0,4,40,6.75,6.750,1.000,1.0,4.0
8,17,641769,641769518,9,2025-06-09,0.0,0.0,1,36,2.75,2.750,1.000,2.0,4.0
9,17,641769,641769518,10,2025-06-16,0.0,0.0,8,35,3.00,3.000,1.000,2.0,4.0
